# Selected-span rule diagnostic: development only

Read archive/claim_syntax_experiments/SELECTED_SPAN_RULE_DIAGNOSTIC_PROTOCOL_v1.md and docs/METHODS.md. Rules were informed by known examples; human-selected spans provide an oracle advantage. This notebook compares target classes (personal killing admission/denial versus contrasts); it does not predict all five annotation fields. No independent accuracy or corpus recall is estimated.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/processed/model_corpus_v1.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the project root or notebooks directory.")
sys.path.insert(0, str(ROOT / "src"))
from run_selected_span_diagnostic import run
comparison, counts, outcomes, manifest = run(ROOT)
print("Human-selected development propositions:", manifest["propositions"])
print("Reference classes:", manifest["reference_counts"])
print("Inputs are selected spans, not automatically extracted units.")
display(outcomes)
display(counts)


## Inspect disagreements and abstentions separately

B1 can avoid wrong predictions by abstaining. Count that cost explicitly. There are only three clear target propositions in these 37 selections, and uncertain reference rows are not ordinary negatives. Do not compare headline accuracy or infer statistical superiority. The common lexicon is held fixed across methods; multiple constraints change together.


In [ ]:
columns = ["claim_id", "execution_number", "evidence_span", "method", "reference_class",
           "prediction", "rule_reason", "outcome"]
issues = comparison.loc[comparison.outcome.isin(["disagreement", "abstention"]), columns]
display(issues)
print("Uncertain reference rows are separate, not silently scored as other:")
display(comparison.loc[comparison.reference_class.eq("reference_uncertain"), columns])


## Return to the whole statement

The worked example is a known imperative already discussed by the owner. To inspect another case, call show_rule_case with its exact claim_id. The unchanged old screen is applied to spans/documents only for cue diagnostics; missing a span cue is not automatically a whole-document retrieval miss.


In [ ]:
def show_rule_case(claim_id):
    selected = comparison.loc[comparison.claim_id.eq(claim_id)]
    if selected.empty:
        raise ValueError("Unknown claim_id.")
    from prepare_confirmed_claims import materialize
    claims, _, _, _ = materialize(ROOT)
    original = claims.set_index("claim_id").loc[claim_id]
    print("Full prepared context:\n", original.model_text)
    print("Selected proposition:", original.evidence_span)
    print("Owner interpretation note:", original.review_note)
    display(selected[columns])
show_rule_case("v3-dev21-024")
